# Data Preprocessing

This notebook was split from the original G'Contest final-round working notebook for faster GitHub review. Raw data and data-loading code are intentionally excluded. Saved outputs are preserved so interviewers can inspect the completed analysis and results without rerunning the private dataset.

**Interview focus:** Transforms the cleaned dataset into model-ready customer-level features, including demographics, time features, and failure history.


# IV. Data Preprocessing

In [157]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, callback
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [158]:
def move_last_succeed_to_end(group):
    # Nếu không có dòng succeed, giữ nguyên thứ tự
    succeed_mask = group['status'] == 'succeed'
    if succeed_mask.any():
        succeed_rows = group[succeed_mask]
        not_succeed_rows = group[~succeed_mask]
        # Đặt succeed cuối cùng (nếu có nhiều) xuống cuối
        return pd.concat([not_succeed_rows, succeed_rows.tail(1)], axis=0)
    else:
        return group

df_sorted = (
    df_cleaned
    .sort_values(['customer_id', 'begin_time'])
    .groupby('customer_id', group_keys=False)
    .apply(move_last_succeed_to_end)
    .reset_index(drop=True)
)


In [159]:
df_model = df_sorted[['begin_time', 'close_time', 'status', 'gender', 'birth_year', 'phone_type', 'stop_point', 'customer_id']].copy()

In [160]:
df_model.isna().sum()

,0
begin_time,0
close_time,0
status,0
gender,0
birth_year,0
phone_type,0
stop_point,0
customer_id,0


## Feature Engineering

### Gender

In [161]:
# Xử lý gender với One-Hot Encoding
gender_ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' để tránh đa cộng tuyến
gender_encoded = gender_ohe.fit_transform(df_model[['gender']])
gender_columns = [f'gender_{cat}' for cat in gender_ohe.categories_[0][1:]]  # Loại bỏ cột đầu tiên
df_gender = pd.DataFrame(gender_encoded, columns=gender_columns, index=df_model.index)

### Age

In [162]:
# Xử lý birth_year và tính age
df_model['age'] = 2025 - df_model['birth_year']

### Phone Type

In [163]:
from sklearn.preprocessing import OneHotEncoder

# Step 1: Lấy mapping từ khách hàng duy nhất
df_customer = df_cleaned.drop_duplicates(subset='customer_id')
phone_counts = df_customer['phone_type'].value_counts(normalize=True)
phone_map = phone_counts[phone_counts >= 0.01].index.tolist()

# Step 2: Gộp phone_type trong toàn bộ dataset theo mapping trên
df_cleaned['phone_type_grouped'] = df_cleaned['phone_type'].apply(lambda x: x if x in phone_map else 'Others')

# Step 3: One-hot encoding
phone_ohe = OneHotEncoder(drop=[['Apple']], sparse_output=False)
# Drop first là Apple vì đây là nhóm phổ biến nhất, giúp mô hình so sánh các nhóm còn lại với chuẩn phổ biến, từ đó dễ giải thích kết quả và tránh đa cộng tuyến.
phone_encoded = phone_ohe.fit_transform(df_cleaned[['phone_type_grouped']])
phone_cols = [f'phone_{cat}' for cat in phone_ohe.categories_[0][1:]]
df_phone = pd.DataFrame(phone_encoded, columns=phone_cols, index=df_cleaned.index)

### Time Features

In [164]:
# Tạo các đặc trưng thời gian
df_model['hour_of_day'] = df_model['begin_time'].dt.hour
df_model['day_of_week'] = df_model['begin_time'].dt.dayofweek
df_model['is_weekend'] = (df_model['day_of_week'] >= 5).astype(int)

In [165]:
# Thêm đặc trưng thời gian trong ngày
def categorize_time(hour):
    if 5 <= hour < 12: return 'morning'
    elif 12 <= hour < 17: return 'afternoon'
    else: return 'evening'
df_model['time_of_day_category'] = df_model['hour_of_day'].apply(categorize_time)
time_ohe = OneHotEncoder(sparse_output=False, drop= 'first')

time_encoded = time_ohe.fit_transform(df_model[['time_of_day_category']])
time_columns = [f'time_{cat}' for cat in time_ohe.categories_[0][1:]]
df_time = pd.DataFrame(time_encoded, columns=time_columns, index=df_model.index)

In [166]:
def is_office_hour(row):
    # day_of_week: Monday=0, ..., Sunday=6
    # office hour: 8h-17h, từ thứ 2 (0) đến thứ 6 (4)
    return 1 if (8 <= row['hour_of_day'] < 17) and (row['day_of_week'] in [0, 1, 2, 3, 4]) else 0

df_model['is_office_hour'] = df_model.apply(is_office_hour, axis=1)

### Historical Attempt Features

In [167]:
# Số thứ tự lần thử mở tài khoản của mỗi khách hàng.
df_model['cumulative_attempts'] = df_model.groupby('customer_id').cumcount() + 1

In [168]:
# Tổng số fail theo lịch sử (không tính record hiện tại)
def calc_total_fail_pending(group):
    fail_hist, pending_hist = [], []
    fail_count, pending_count = 0, 0
    for i, row in group.iterrows():
        fail_hist.append(fail_count)
        pending_hist.append(pending_count)
        if row['status'] == 'fail':
            fail_count += 1
        if row['status'] == 'pending':
            pending_count += 1
    return pd.DataFrame({'total_fail_past': fail_hist, 'total_pending_past': pending_hist}, index=group.index)

# Áp dụng theo từng customer_id
fail_pending_hist = df_model.groupby('customer_id', group_keys=False).apply(calc_total_fail_pending)

# Kết hợp lại vào df_model
df_model = pd.concat([df_model.reset_index(drop=True), fail_pending_hist.reset_index(drop=True)], axis=1)

### Failure History and Stop-Point Encoding

In [169]:
# Đếm fail ở từng stop_point
fail_pending_count = [
    'otp-failed', 'data-privacy-rejected', 'ocr-failed',
    'selfie-failed', 'contract-rejected', 'check-failed',
]

def calc_fail_count_past(group):
    # Chuẩn hóa stop_point là string (nếu đã map về số, phải đổi lại)
    counts = {f'fail_pending_count_{sp}': [] for sp in fail_pending_count}
    history = {sp: 0 for sp in fail_pending_count}
    for i, row in group.iterrows():
        for sp in fail_pending_count:
            counts[f'fail_pending_count_{sp}'].append(history[sp])
        # Cập nhật lịch sử sau khi lưu vào counts
        if row['status'] in ['fail', 'pending'] and row['stop_point'] in fail_pending_count:
            history[row['stop_point']] += 1
    return pd.DataFrame(counts, index=group.index)

# Tính fail count lịch sử cho từng dòng của từng khách
fail_pending_count_past = df_model.groupby('customer_id', group_keys=False).apply(calc_fail_count_past)

# Kết hợp vào df_model (vị trí dòng phải khớp, thường là .reset_index(drop=True))
df_model = pd.concat([df_model.reset_index(drop=True), fail_pending_count_past.reset_index(drop=True)], axis=1)

In [170]:
# Mã hóa stop_point theo trình tự thủ công
stop_point_mapping = {
    'otp-failed': 0,
    'data-privacy-rejected': 1,
    'ocr-failed': 2,
    'selfie-failed': 3,
    'contract-rejected': 4,
    'check-failed': 5,
    'account-creation': 6
}

df_model['stop_point_mapped'] = df_model['stop_point'].map(stop_point_mapping)

### Tổng hợp các biến mới tạo

In [171]:
# Kết hợp đặc trưng đã mã hóa, lịch sử, fail stop_point

# MỚI: Bỏ mode_stop_point_past và previous_stop_point do encode không phù hợp

df_model_combine = pd.concat([
    df_model[['customer_id','age', 'status', 'hour_of_day', 'day_of_week', 'cumulative_attempts',
              'stop_point_mapped', 'total_fail_past', 'total_pending_past', 'is_office_hour','is_weekend',
              'fail_pending_count_otp-failed',
              'fail_pending_count_data-privacy-rejected',
              'fail_pending_count_ocr-failed',
              'fail_pending_count_selfie-failed',
              'fail_pending_count_contract-rejected',
              'fail_pending_count_check-failed',]],
    df_gender, df_phone, df_time
], axis=1)
